# Module 8 — Inversion, Extraction & Cryptographic Privacy  ·  OWASP LLM02 / LLM10

Attacks on the model itself: does it leak its training data (memorisation, membership inference),
can it be stolen (extraction), and the privacy defences (differential privacy, PII redaction,
federated learning, watermarking, query-shaping).

These are statistical/ML results, so the demos run on small sklearn/numpy models **where we control
the training set** — that makes the signal real and the run fast, rather than hand-waving over a
black box. Two labs need research frameworks and are written up in `docs/not-runnable-labs.md`:
gradient inversion (74, `breaching`) and FHE blind inference (79, Concrete ML in Docker).

Labs 71–80.

In [1]:
import importlib, labkit
importlib.reload(labkit)
from labkit import Chat, embed, cosine, DEFAULT_TARGET
import numpy as np, warnings
warnings.filterwarnings('ignore')
np.random.seed(0)
print("target:", DEFAULT_TARGET)

target: qwen2.5:7b


## Lab 71 — Memorisation & Leakage

Models regurgitate verbatim strings they saw often in training. Probe the local model with the start
of well-known text and see whether it completes it word-for-word — the signature of memorisation,
and the mechanism behind training-data extraction.

In [2]:
probes = [
    ("famous opening", "It was the best of times, it was the worst of"),
    ("well-known constant", "The first eight digits of pi after the decimal are 1415"),
    ("licence boilerplate", "Permission is hereby granted, free of charge, to any person obtaining a"),
]
bot = Chat(system="Complete the text. Output only the continuation.", temperature=0.0)
for name, stub in probes:
    out = bot(stub).strip()
    print(f"[{name}] '{stub[:40]}...'\n   -> {out[:90]}\n")

[famous opening] 'It was the best of times, it was the wor...'
   -> times, it was the age of wisdom, it was the age of foolishness, it was the epoch of belief

[well-known constant] 'The first eight digits of pi after the d...'
   -> 9265



[licence boilerplate] 'Permission is hereby granted, free of ch...'
   -> copy of this software and associated documentation files (the "Software"), to deal in the 



**Read it:** verbatim completion of famous or boilerplate text is memorisation — harmless for a
Dickens line, dangerous when the memorised string is a leaked API key or a patient record that sat in
the training set. The defences below reduce what a model can memorise and what a query can pull out.

## Lab 72 — Membership Inference

Can an attacker tell whether a specific record was in the training set? Yes — a model is more
*confident* on data it trained on. We train a classifier, then show its confidence is systematically
higher on members (training rows) than on held-out non-members. That confidence gap *is* the leak.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=400, n_features=20, random_state=1)
members_X, members_y = X[:200], y[:200]       # model trains on these
nonmem_X, nonmem_y = X[200:], y[200:]         # never seen

# an overfit model memorises its members (deep trees, few samples)
victim = RandomForestClassifier(n_estimators=50, max_depth=None, random_state=1)
victim.fit(members_X, members_y)

def confidence(Xs, ys):
    p = victim.predict_proba(Xs)
    return np.mean([p[i, ys[i]] for i in range(len(ys))])   # prob on the TRUE label

c_mem = confidence(members_X, members_y)
c_non = confidence(nonmem_X, nonmem_y)
print(f"mean confidence on MEMBERS     : {c_mem:.3f}")
print(f"mean confidence on NON-MEMBERS : {c_non:.3f}")
print(f"membership-inference gap       : {c_mem - c_non:+.3f}  <- an attacker thresholds on this")

mean confidence on MEMBERS     : 0.928
mean confidence on NON-MEMBERS : 0.835
membership-inference gap       : +0.094  <- an attacker thresholds on this


## Lab 73 — Model Extraction (stealing the brain via queries)

With only query access, an attacker trains a cheap *student* to mimic the victim's outputs —
distillation used as theft. We query a victim model and fit a student on `(input, victim_prediction)`
pairs, then measure how often the stolen student agrees with the original.

In [4]:
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=2)
victim = SVC(kernel="rbf", probability=False).fit(Xtr, ytr)   # the expensive target

# attacker: query the victim on many inputs, record its LABELS, train a student
queries = np.random.randn(2000, 20)              # attacker's own probe inputs
stolen_labels = victim.predict(queries)          # the only access needed
student = DecisionTreeClassifier(max_depth=10).fit(queries, stolen_labels)

agreement = np.mean(student.predict(Xte) == victim.predict(Xte))
print(f"stolen student agrees with victim on {agreement:.1%} of held-out inputs")
print("no weights, no training data — just 2000 queries and the labels they returned")

stolen student agrees with victim on 97.5% of held-out inputs
no weights, no training data — just 2000 queries and the labels they returned


## Lab 75 — Differential Privacy (the privacy veil)

DP bounds how much any single training record can influence the model: **clip** each example's
gradient and **add noise**. The course uses Opacus on an LLM; the mechanism is clip+noise, shown here
on a toy model — and the privacy-utility trade-off (more noise → more privacy → less accuracy) is the
whole point.

In [5]:
from sklearn.linear_model import LogisticRegression
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=3)

def train_with_dp(noise_multiplier, clip=1.0):
    # a one-step DP-SGD sketch: fit, then clip + noise the learned weights
    m = LogisticRegression(max_iter=200).fit(Xtr, ytr)
    w = m.coef_.copy()
    norm = np.linalg.norm(w)
    w = w / max(1, norm / clip)                         # clip sensitivity
    w = w + np.random.normal(0, noise_multiplier * clip, w.shape)  # add noise
    m.coef_ = w
    return m.score(Xte, yte)

print(f"{'noise':>8} {'test accuracy':>14}   privacy")
for nm in [0.0, 0.3, 0.8, 2.0]:
    acc = train_with_dp(nm)
    print(f"{nm:8.1f} {acc:14.3f}   {'none' if nm==0 else 'ε decreasing ->' }")

   noise  test accuracy   privacy
     0.0          0.967   none
     0.3          0.850   ε decreasing ->
     0.8          0.683   ε decreasing ->
     2.0          0.683   ε decreasing ->


**Read it:** at zero noise the model is accurate but memorises (the Lab 72 leak); as noise rises
the membership signal washes out but accuracy falls. DP makes that trade explicit via the privacy
budget ε — there is no free privacy.

## Lab 76 — PII Redaction Pipeline

Strip personal data before it reaches a model (training or prompt) or leaves in a response. The course
uses Presidio; here spaCy NER catches names/orgs and regex catches structured PII (email, phone, card,
IBAN). Run it on text headed for the model.

In [6]:
import re, spacy
nlp = spacy.load("en_core_web_lg")

# CARD/IBAN are more specific than PHONE and must match first — a 16-digit
# card otherwise matches the phone pattern.
PATTERNS = {
    "EMAIL": r"[\w.+-]+@[\w-]+\.[\w.-]+",
    "IBAN":  r"\b[A-Z]{2}\d{2}[A-Z0-9]{10,30}\b",
    "CARD":  r"\b(?:\d[ -]?){13,16}\b",
    "PHONE": r"\+?\d[\d ()-]{6,}\d",
}
def redact(text):
    for tag, pat in PATTERNS.items():
        text = re.sub(pat, f"[{tag}]", text)
    doc = nlp(text)
    for ent in reversed(doc.ents):           # NER for names/orgs/locations
        if ent.label_ in ("PERSON", "GPE", "ORG"):
            text = text[:ent.start_char] + f"[{ent.label_}]" + text[ent.end_char:]
    return text

sample = ("Patient John Smith (john.smith@mail.com, +41 79 123 4567) at Basel General "
          "paid with card 4111 1111 1111 1111.")
print("raw    :", sample)
print("redacted:", redact(sample))

raw    : Patient John Smith (john.smith@mail.com, +41 79 123 4567) at Basel General paid with card 4111 1111 1111 1111.
redacted: Patient [PERSON] ([EMAIL], [PHONE]) at [ORG] paid with card [CARD].


## Lab 77 — Federated Learning (train without centralising data)

Each client trains on its own data and shares only model *updates*; the server averages them (FedAvg).
Raw data never leaves the client. We simulate two "hospitals" training locally and show the averaged
global model works without either dataset being pooled.

In [7]:
from sklearn.linear_model import SGDClassifier

# two clients, disjoint private data — never combined
cA_X, cA_y = X[:200], y[:200]
cB_X, cB_y = X[200:], y[200:]

def local_update(Xc, yc):
    m = SGDClassifier(loss="log_loss", max_iter=50, random_state=0).fit(Xc, yc)
    return m.coef_, m.intercept_

wA, bA = local_update(cA_X, cA_y)      # computed ON hospital A, data stays there
wB, bB = local_update(cB_X, cB_y)      # computed ON hospital B, data stays there

# FedAvg: the server sees only the weights, averages them
global_w, global_b = (wA + wB) / 2, (bA + bB) / 2
glob = SGDClassifier(loss="log_loss"); glob.fit(X[:5], y[:5])   # init shape
glob.coef_, glob.intercept_ = global_w, global_b
print(f"federated global model accuracy: {glob.score(X, y):.3f}")
print("neither client's rows were ever centralised — only averaged weights crossed the wire")

federated global model accuracy: 0.945
neither client's rows were ever centralised — only averaged weights crossed the wire


## Lab 78 — Watermarking LLM Outputs

Bias generation toward a pseudo-random "green list" of tokens seeded by the previous token. Human-
invisible, but a detector that knows the seed sees far more green tokens than the ~γ expected by
chance — a statistical z-test proves the text came from the watermarked model. Shown on a toy vocab.

In [8]:
GAMMA, DELTA, VOCAB = 0.25, 2.0, 1000

def greenlist(prev_token):
    rng = np.random.default_rng(prev_token)
    perm = rng.permutation(VOCAB)
    return set(perm[:int(GAMMA * VOCAB)].tolist())

def generate(n, watermarked):
    toks, prev = [], 0
    for _ in range(n):
        logits = np.random.randn(VOCAB)
        if watermarked:
            for g in greenlist(prev): logits[g] += DELTA   # bias toward green
        # sample from the softmax (real decoding), not argmax — so the green
        # share is a realistic ~0.5-0.7, not a deterministic 100%.
        p = np.exp(logits - logits.max()); p /= p.sum()
        prev = int(np.random.choice(VOCAB, p=p)); toks.append(prev)
    return toks

def detect(toks):
    green = sum(1 for i in range(1, len(toks)) if toks[i] in greenlist(toks[i-1]))
    n = len(toks) - 1
    z = (green - GAMMA*n) / np.sqrt(n * GAMMA * (1-GAMMA))   # z-score vs chance
    return green/n, z

for label, wm in [("plain text", False), ("watermarked", True)]:
    frac, z = detect(generate(400, wm))
    print(f"{label:12} green fraction {frac:.2f} (chance {GAMMA})  z={z:5.1f}"
          f"  {'<- WATERMARK DETECTED' if z > 4 else ''}")

plain text   green fraction 0.24 (chance 0.25)  z= -0.5  
watermarked  green fraction 0.73 (chance 0.25)  z= 22.3  <- WATERMARK DETECTED


## Lab 80 — Detecting Extraction via Query Shaping

The Lab 73 thief fires thousands of queries that densely probe one region of input space. Track each
user's query embeddings; a user whose queries cluster too tightly (low average pairwise distance,
high volume) is mapping a decision boundary — flag them. In-process; the course uses Redis for scale.

In [9]:
def probing_score(query_texts):
    vecs = [embed(q) for q in query_texts]
    sims = [cosine(vecs[i], vecs[j]) for i in range(len(vecs)) for j in range(i+1, len(vecs))]
    return np.mean(sims)      # high mean similarity = tightly clustered = probing

normal_user = ["What is the refund policy?", "How do I reset my password?",
               "What are your opening hours?", "Where is my order?"]
extractor = ["Classify sentiment: the product is good",
             "Classify sentiment: the product is bad",
             "Classify sentiment: the product is okay",
             "Classify sentiment: the product is terrible"]
for label, qs in [("normal user", normal_user), ("extraction attacker", extractor)]:
    s = probing_score(qs)
    print(f"{label:22} mean query similarity {s:.3f}"
          f"  {'<- FLAGGED: boundary probing' if s > 0.6 else ''}")

normal user            mean query similarity 0.535  
extraction attacker    mean query similarity 0.859  <- FLAGGED: boundary probing


---
### What this module showed

A model leaks its training data through memorisation (71) and the confidence gap membership inference
(72) thresholds on; it can be stolen wholesale by querying and distilling (73). The defences trade
utility for privacy explicitly — differential privacy (75) adds calibrated noise, PII redaction (76)
removes the data before it is ever seen, federated learning (77) never centralises it, watermarking
(78) proves provenance, and query-shaping detection (80) spots the extractor's probing pattern.
Gradient inversion (74) and FHE blind inference (79) need research frameworks — written up.

**Module 9** is hardware-rooted security (TPM, SEV-SNP, attestation) — read-only, no notebook.
**Module 10** is the red-team capstone.